# BOA push service: dev

Run this notebook using the Python environment installed from this PR branch:

```powershell
python -m pip install -e ".[dev]"
```

Select that environment as the notebook kernel, then run the cells in order. This notebook explicitly selects **dev/staging**. The compatible BOA backend must be deployed there, and the account must have live GB BOA API access.
Credentials come from `LCPDELTA_ENACT_USERNAME` and `LCPDELTA_ENACT_API_KEY`, or interactive prompts. The API key prompt is hidden.


In [ ]:
import os

os.environ["ENACT_ENV"] = "dev"

import asyncio
import logging
from collections import Counter, deque
from datetime import datetime, timezone
from getpass import getpass
from threading import Lock

import pandas as pd
from IPython.display import display

from lcp_delta.common.constants import PUSH_SERVICE_BASE_URL_DEV
from lcp_delta.enact import BoaDPSHelper, BoaPush
from lcp_delta.enact.loader import get_base_endpoints

base_url = get_base_endpoints().PUSH_SERVICE_BASE_URL
assert base_url == PUSH_SERVICE_BASE_URL_DEV, "Expected the dev push service"
print(f"Dev SignalR endpoint: {base_url}/dataHub")
logging.basicConfig(level=logging.INFO)

LISTEN_SECONDS = 60

In [ ]:
username = os.getenv("LCPDELTA_ENACT_USERNAME") or input("Enact username: ").strip()
api_key = os.getenv("LCPDELTA_ENACT_API_KEY") or getpass("Enact public API key: ")
if not username or not api_key:
    raise ValueError("A username and public API key are required")

## Subscribe and capture

The next cell subscribes once, listens for `LISTEN_SECONDS`, and closes the connection in `finally`, including if the cell is interrupted. Rerun it to start a fresh capture. Connection and heartbeat messages appear in the output.

One callback receives accepted and on-table changes/removals and prints their rows immediately, with the received time, settlement date, period and revision. The notebook counts every delivered batch and row, and retains only the latest 25 batches for inspection. The backend maintains the 24-hour window from the current settlement period.


In [ ]:
COLLECTIONS = (
    "accepted_bids",
    "accepted_offers",
    "removed_accepted_bids",
    "removed_accepted_offers",
    "table_bids",
    "table_offers",
    "removed_table_bids",
    "removed_table_offers",
)
recent_batches = deque(maxlen=25)
row_counts = Counter()
capture_lock = Lock()
batch_count = 0


def on_update(batch: BoaPush):
    global batch_count
    received_at = datetime.now(timezone.utc)
    with capture_lock:
        batch_count += 1
        recent_batches.append((received_at, batch))
        row_counts.update({name: len(getattr(batch, name)) for name in COLLECTIONS})
    rows = [{"collection": name, **row} for name in COLLECTIONS for row in getattr(batch, name)]
    table = pd.DataFrame(rows).reindex(
        columns=["collection", "plantId", "bodId", "volume", "bidPrice", "offerPrice", "acceptedTime"]
    )
    print(
        f"\nReceived (UTC): {received_at.isoformat()} | "
        f"Settlement date: {batch.date} | Period: {batch.period} | Revision: {batch.revision}\n"
        f"{table.to_string(index=False)}",
        flush=True,
    )


stream = BoaDPSHelper(
    username,
    api_key,
    callback=on_update,
    auto_connect=False,
    callback_queue_maxsize=1000,
    heartbeat_timeout_seconds=30,
)
try:
    group = await stream.async_subscribe_to_boa_updates(timeout=30)
    assert group == "boaDashboardUpdated"
    assert stream.is_connected
    print(f"Subscribed to {group} on dev; listening for {LISTEN_SECONDS} seconds...")
    await asyncio.sleep(LISTEN_SECONDS)
finally:
    await stream.async_close(wait_for_callbacks=True)

print(f"Connection closed. Received {batch_count} batches and {sum(row_counts.values())} row changes/removals.")

## Inspect received pushes

The table shows counts for all eight collections, followed by up to 100 rows from the latest batch. `collection` distinguishes changes from removals. The rows are the cleaned public callback data. Inspect `latest_batch` directly for the complete latest batch.


In [ ]:
with capture_lock:
    captured = list(recent_batches)
    totals = row_counts.copy()

display(pd.DataFrame([{"collection": name, "row_count": totals[name]} for name in COLLECTIONS]))

if captured:
    received_at, latest_batch = captured[-1]
    print(f"Received (UTC): {received_at.isoformat()}")
    print(f"Settlement date: {latest_batch.date} | Period: {latest_batch.period} | Revision: {latest_batch.revision}")
    rows = [{"collection": name, **row} for name in COLLECTIONS for row in getattr(latest_batch, name)]
    display(pd.DataFrame(rows).head(100))
else:
    print("No actual BOA changes received. Increase LISTEN_SECONDS and rerun the capture cell.")

This is a manual live-service check. A successful subscription confirms that dev accepted the join; receiving data additionally requires an actual BOA change during the capture window. No initial snapshot or replay is requested.

Predicted rows are ignored, including predicted removals, and cannot remove previously delivered actual rows. `reloadRequired` and `previousRevision` are ignored by the helper. Callback collections are deltas, not full period snapshots.

The notebook is committed without outputs. Clear outputs before committing a local live run.
